<a href="https://colab.research.google.com/github/gadekarvishal08-cloud/IN226005202_NLP/blob/main/Task3_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
chat_history_ids = None

def chatbot(user_input):
    global chat_history_ids

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Create attention mask
    new_attention_mask = torch.ones_like(new_input_ids)

    # Append history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        attention_mask = torch.ones_like(bot_input_ids)
    else:
        bot_input_ids = new_input_ids
        attention_mask = new_attention_mask

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,   # ✅ FIX HERE
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response

In [9]:
chat_history_ids = None

def chatbot(user_input):
    global chat_history_ids

    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Attention mask
    attention_mask = torch.ones_like(bot_input_ids)

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response

In [10]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop or 'clear' to reset chat.")

while True:
    user_input = input("You: ")

    if user_input.lower() in ['exit', 'quit']:
        print("Chatbot: Goodbye! 👋")
        break

    if user_input.lower() == "clear":
        chat_history_ids = None
        print("Chatbot: Conversation reset.")
        continue

    response = chatbot(user_input)
    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop or 'clear' to reset chat.
You: hello
Chatbot: hi i dont know you but i love you
You: i am vishal
Chatbot: hi vishal


KeyboardInterrupt: Interrupted by user